# Pure Structured KNN Classifier (Cybersecurity Dataset)
This notebook builds a simple non-parametric KNN classifier using only structured features (numerical + categorical). No TF-IDF or text processing is used.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# 1) Load data
df = pd.read_csv("cybersecurity_attacks.csv")

TARGET = "Attack Type"   # Change this if your label column has a different name
df = df.dropna(subset=[TARGET]).copy()

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str)

print("Dataset shape:", df.shape)


In [ ]:
# 2) Identify structured columns
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))


In [ ]:
# 3) Preprocessing (structured only)
numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", categorical_preprocess, cat_cols),
    ]
)


In [ ]:
# 4) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:
# 5) Build KNN model (non-parametric)
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("knn", KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
        metric="euclidean"
    )),
])


In [ ]:
# 6) Train model
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, pred))
